# WikiArt Style Classification — Test 6 (Highest Accuracy Notebook)

This notebook is made to push accuracy as high as possible, based on what worked best in your tests 1 to 5.

### What is improved here (easy words)
- Strong model: **ViT-Base 384** pretrained weights
- Two-step training: first train the head, then fine-tune full model
- Better regularization: MixUp + CutMix + label smoothing
- Better stability: EMA model averaging + gradient clipping
- Better optimization: warmup + cosine learning rate schedule
- Better class balance: weighted sampler
- Better final score: optional TTA (flip test-time augmentation)

> Goal: beat your current best (~69.3% validation top-1) and push over 70%.

## 1) Setup and Imports

Run this cell first. It imports everything, sets random seeds, and picks GPU automatically.

In [1]:
import copy
import math
import os
import random
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

try:
    import timm
except ImportError:
    raise ImportError("Please install timm first: pip install timm")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Faster matmul on modern GPUs
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2) Full High-Accuracy Pipeline Code

This cell contains the full training pipeline (same strategy as the best script, but now directly in notebook form).

In [2]:
@dataclass
class TrainConfig:
    model_name: str = "vit_base_patch16_384.augreg_in21k_ft_in1k"
    image_size: int = 384
    batch_size: int = 12
    effective_batch_size: int = 48
    head_epochs: int = 2
    ft_epochs: int = 24
    patience: int = 8
    warmup_epochs: int = 2
    head_lr: float = 6e-4
    ft_backbone_lr: float = 1e-5
    ft_head_lr: float = 3e-5
    weight_decay: float = 8e-5
    mixup_alpha: float = 0.4
    cutmix_alpha: float = 0.8
    mix_probability: float = 0.75
    label_smoothing: float = 0.12
    ema_decay: float = 0.9997
    grad_clip_norm: float = 1.0
    tta: bool = True
    use_weighted_sampler: bool = True
    seed: int = 42
    num_workers: int = 0


class WikiArtStyleDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, image_root: Path, transform=None):
        self.rows = rows.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows.iloc[idx]
        image_path = self.image_root / row["relative_path"]
        label = int(row["label"])
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9997):
        self.decay = decay
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.ema_model.state_dict().items():
            if k in msd:
                model_v = msd[k].detach()
                if torch.is_floating_point(v):
                    v.mul_(self.decay).add_(model_v, alpha=1.0 - self.decay)
                else:
                    v.copy_(model_v)


class SoftTargetCrossEntropyWithLabelSmoothing(nn.Module):
    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = logits.size(1)
        if targets.dim() == 1:
            with torch.no_grad():
                true_dist = torch.zeros_like(logits)
                true_dist.fill_(self.smoothing / (n_classes - 1))
                true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
            targets = true_dist
        log_probs = torch.log_softmax(logits, dim=1)
        return torch.mean(torch.sum(-targets * log_probs, dim=1))


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def discover_paths(project_root: Path) -> Tuple[Path, Path, Path]:
    wikiart_dir = project_root / "datasets" / "Wikiart"
    train_csv = wikiart_dir / "style_train.csv"
    val_csv = wikiart_dir / "style_val.csv"
    if not train_csv.exists() or not val_csv.exists():
        raise FileNotFoundError("Could not find style_train.csv and style_val.csv in datasets/Wikiart")
    return wikiart_dir, train_csv, val_csv


def load_style_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["relative_path", "label"])
    df["label"] = df["label"].astype(int)
    return df


def make_eval_split(val_df: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    grouped = val_df.groupby("label", group_keys=False)
    eval_test = grouped.apply(lambda x: x.sample(frac=0.5, random_state=seed) if len(x) > 1 else x)
    eval_val = val_df.drop(eval_test.index)
    eval_test = eval_test.reset_index(drop=True)
    eval_val = eval_val.reset_index(drop=True)
    if len(eval_val) == 0:
        eval_val = val_df.sample(frac=0.5, random_state=seed).reset_index(drop=True)
        eval_test = val_df.drop(eval_val.index).reset_index(drop=True)
    return eval_val, eval_test


def filter_existing_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    keep_mask = df["relative_path"].map(lambda p: (image_root / p).exists())
    missing_count = int((~keep_mask).sum())
    if missing_count > 0:
        print(f"[{split_name}] Filtered out {missing_count} missing files.")
    return df.loc[keep_mask].reset_index(drop=True)


def build_transforms(image_size: int):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
        transforms.RandomErasing(p=0.18, scale=(0.02, 0.16), ratio=(0.3, 3.3), value="random"),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tfms, eval_tfms


def create_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    class_counts = np.bincount(labels)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]
    sample_weights = np.sqrt(sample_weights)
    sample_weights = sample_weights / sample_weights.mean()
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


def mixup_cutmix(inputs, targets, alpha_mixup, alpha_cutmix, p, num_classes):
    if np.random.rand() > p:
        return inputs, torch.nn.functional.one_hot(targets, num_classes=num_classes).float()

    batch_size = inputs.size(0)
    indices = torch.randperm(batch_size, device=inputs.device)
    targets_onehot = torch.nn.functional.one_hot(targets, num_classes=num_classes).float()
    shuffled_targets = targets_onehot[indices]

    use_cutmix = alpha_cutmix > 0 and np.random.rand() < 0.5
    if use_cutmix:
        lam = np.random.beta(alpha_cutmix, alpha_cutmix)
        h, w = inputs.size(2), inputs.size(3)
        cut_rat = np.sqrt(1.0 - lam)
        cut_w = int(w * cut_rat)
        cut_h = int(h * cut_rat)

        cx = np.random.randint(w)
        cy = np.random.randint(h)

        x1 = np.clip(cx - cut_w // 2, 0, w)
        x2 = np.clip(cx + cut_w // 2, 0, w)
        y1 = np.clip(cy - cut_h // 2, 0, h)
        y2 = np.clip(cy + cut_h // 2, 0, h)

        mixed = inputs.clone()
        mixed[:, :, y1:y2, x1:x2] = inputs[indices, :, y1:y2, x1:x2]
        lam_adjusted = 1.0 - ((x2 - x1) * (y2 - y1) / (w * h))
        soft_targets = targets_onehot * lam_adjusted + shuffled_targets * (1.0 - lam_adjusted)
        return mixed, soft_targets

    lam = np.random.beta(alpha_mixup, alpha_mixup) if alpha_mixup > 0 else 1.0
    mixed = lam * inputs + (1.0 - lam) * inputs[indices]
    soft_targets = lam * targets_onehot + (1.0 - lam) * shuffled_targets
    return mixed, soft_targets


def topk_accuracy(logits, targets, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        _, pred = logits.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(targets.view(1, -1).expand_as(pred))
        out = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            out.append(correct_k / targets.size(0))
        return out


def run_epoch(model, loader, criterion, optimizer, device, scaler, num_classes, cfg, ema, is_train, accum_steps):
    model.train(is_train)
    total_loss = total_top1 = total_top5 = 0.0
    total_samples = 0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            if is_train:
                images_mixed, soft_targets = mixup_cutmix(
                    images, targets, cfg.mixup_alpha, cfg.cutmix_alpha, cfg.mix_probability, num_classes
                )
                logits = model(images_mixed)
                loss = criterion(logits, soft_targets)
            else:
                logits = model(images)
                loss = criterion(logits, targets)

            loss = loss / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if step % accum_steps == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model)

        with torch.no_grad():
            top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_samples += bs
        total_loss += loss.item() * accum_steps * bs
        total_top1 += float(top1.item()) * bs
        total_top5 += float(top5.item()) * bs

    return {
        "loss": total_loss / max(1, total_samples),
        "top1": total_top1 / max(1, total_samples),
        "top5": total_top5 / max(1, total_samples),
    }


@torch.no_grad()
def evaluate_with_tta(model, loader, criterion, device, use_tta):
    model.eval()
    total_loss = total_top1 = total_top5 = 0.0
    total_samples = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(images)
        if use_tta:
            logits_flip = model(torch.flip(images, dims=[3]))
            logits = (logits + logits_flip) / 2.0

        loss = criterion(logits, targets)
        top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_samples += bs
        total_loss += float(loss.item()) * bs
        total_top1 += float(top1.item()) * bs
        total_top5 += float(top5.item()) * bs

    return {
        "loss": total_loss / max(1, total_samples),
        "top1": total_top1 / max(1, total_samples),
        "top5": total_top5 / max(1, total_samples),
    }


def freeze_backbone(model, freeze):
    for p in model.parameters():
        p.requires_grad = not freeze
    if hasattr(model, "head"):
        for p in model.head.parameters():
            p.requires_grad = True


def get_trainable_parameters(model, backbone_lr, head_lr, weight_decay):
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in name for k in ["head", "fc", "classifier"]):
            head_params.append(p)
        else:
            backbone_params.append(p)

    out = []
    if backbone_params:
        out.append({"params": backbone_params, "lr": backbone_lr, "weight_decay": weight_decay})
    if head_params:
        out.append({"params": head_params, "lr": head_lr, "weight_decay": weight_decay})
    return out


def build_scheduler(optimizer, warmup_epochs, total_epochs):
    warmup_epochs = max(0, warmup_epochs)
    total_epochs = max(1, total_epochs)
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.12, total_iters=warmup_epochs)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_epochs - warmup_epochs))
        return torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs])
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)


def fit(project_root: Path, cfg: TrainConfig):
    set_seed(cfg.seed)
    Image.MAX_IMAGE_PIXELS = None

    wikiart_dir, train_csv, val_csv = discover_paths(project_root)
    train_df = filter_existing_rows(load_style_csv(train_csv), wikiart_dir, "train")
    val_df = filter_existing_rows(load_style_csv(val_csv), wikiart_dir, "val")

    eval_val_df, eval_test_df = make_eval_split(val_df, seed=cfg.seed)
    num_classes = int(max(train_df["label"].max(), val_df["label"].max()) + 1)

    train_tfms, eval_tfms = build_transforms(cfg.image_size)

    train_ds = WikiArtStyleDataset(train_df, wikiart_dir, transform=train_tfms)
    val_ds = WikiArtStyleDataset(eval_val_df, wikiart_dir, transform=eval_tfms)
    test_ds = WikiArtStyleDataset(eval_test_df, wikiart_dir, transform=eval_tfms)

    sampler = create_weighted_sampler(train_df["label"].values) if cfg.use_weighted_sampler else None
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        sampler=sampler,
        shuffle=(sampler is None),
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=True,
    )
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
    print(f"Classes: {num_classes}")

    model = timm.create_model(cfg.model_name, pretrained=True, num_classes=num_classes).to(device)

    eval_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    train_criterion = SoftTargetCrossEntropyWithLabelSmoothing(smoothing=cfg.label_smoothing)

    accum_steps = max(1, math.ceil(cfg.effective_batch_size / cfg.batch_size))
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    ema = ModelEMA(model, decay=cfg.ema_decay)

    best_val_top1 = -1.0
    best_epoch = -1
    patience_left = cfg.patience
    history = []

    models_dir = project_root / "models"
    results_dir = project_root / "models" / "results"
    models_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = models_dir / "wikiart_test6_best.pt"

    epoch_counter = 0

    for stage_name, stage_epochs, freeze_flag, bb_lr, hd_lr in [
        ("head-only", cfg.head_epochs, True, cfg.head_lr, cfg.head_lr),
        ("fine-tune", cfg.ft_epochs, False, cfg.ft_backbone_lr, cfg.ft_head_lr),
    ]:
        freeze_backbone(model, freeze=freeze_flag)
        optimizer = torch.optim.AdamW(get_trainable_parameters(model, bb_lr, hd_lr, cfg.weight_decay))
        scheduler = build_scheduler(optimizer, min(cfg.warmup_epochs, stage_epochs), stage_epochs)

        for _ in range(stage_epochs):
            epoch_counter += 1

            train_metrics = run_epoch(
                model=model,
                loader=train_loader,
                criterion=train_criterion,
                optimizer=optimizer,
                device=device,
                scaler=scaler,
                num_classes=num_classes,
                cfg=cfg,
                ema=ema,
                is_train=True,
                accum_steps=accum_steps,
            )

            eval_model = ema.ema_model
            val_metrics = evaluate_with_tta(eval_model, val_loader, eval_criterion, device, cfg.tta)
            scheduler.step()

            lr_now = optimizer.param_groups[0]["lr"]
            history.append({
                "epoch": epoch_counter,
                "stage": stage_name,
                "lr": lr_now,
                "train_loss": train_metrics["loss"],
                "train_top1": train_metrics["top1"],
                "train_top5": train_metrics["top5"],
                "val_loss": val_metrics["loss"],
                "val_top1": val_metrics["top1"],
                "val_top5": val_metrics["top5"],
            })

            print(
                f"Epoch {epoch_counter:02d} | {stage_name:9s} | lr={lr_now:.2e} | "
                f"train_acc={train_metrics['top1']:.4f} | val_acc={val_metrics['top1']:.4f}"
            )

            if val_metrics["top1"] > best_val_top1:
                best_val_top1 = val_metrics["top1"]
                best_epoch = epoch_counter
                patience_left = cfg.patience
                torch.save({
                    "model_state": copy.deepcopy(eval_model.state_dict()),
                    "config": asdict(cfg),
                    "best_val_top1": best_val_top1,
                    "best_epoch": best_epoch,
                    "num_classes": num_classes,
                    "model_name": cfg.model_name,
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                }, checkpoint_path)
            else:
                patience_left -= 1

            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch_counter}. Best epoch: {best_epoch}")
                break

        if patience_left <= 0:
            break

    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state["model_state"])

    final_val = evaluate_with_tta(model, val_loader, eval_criterion, device, cfg.tta)
    final_test = evaluate_with_tta(model, test_loader, eval_criterion, device, cfg.tta)

    history_df = pd.DataFrame(history)
    history_csv = results_dir / "wikiart_test6_history.csv"
    history_df.to_csv(history_csv, index=False)

    summary_row = {
        "notebook": "wikiart_style_classification_test6_max_accuracy.ipynb",
        "exists": True,
        "model_name": cfg.model_name,
        "val_loss": final_val["loss"],
        "val_top1": final_val["top1"],
        "val_top5": final_val["top5"],
        "test_loss": final_test["loss"],
        "test_top1": final_test["top1"],
        "test_top5": final_test["top5"],
        "best_val_top1": best_val_top1,
        "best_epoch": best_epoch,
        "experiment": "test6",
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    summary_csv = results_dir / "wikiart_tests_1_to_6_summary.csv"
    prior_summary_csv = results_dir / "wikiart_tests_1_to_5_summary.csv"

    rows = []
    if prior_summary_csv.exists():
        rows.extend(pd.read_csv(prior_summary_csv).to_dict(orient="records"))
    rows = [r for r in rows if str(r.get("experiment", "")).lower() != "test6"]
    rows.append(summary_row)
    pd.DataFrame(rows).sort_values("experiment").to_csv(summary_csv, index=False)

    print("\nFinal evaluation")
    print(f"Validation -> loss: {final_val['loss']:.4f}, top1: {final_val['top1']:.4f}, top5: {final_val['top5']:.4f}")
    print(f"Test       -> loss: {final_test['loss']:.4f}, top1: {final_test['top1']:.4f}, top5: {final_test['top5']:.4f}")
    print(f"Best checkpoint: {checkpoint_path}")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val_top1: {best_val_top1:.4f}")
    print(f"Saved history: {history_csv}")
    print(f"Saved summary: {summary_csv}")

    return history_df, final_val, final_test

In [3]:
# Progress-print version of run_epoch (overrides previous function)
# It prints status every N batches, so you can see training is moving.

def run_epoch(model, loader, criterion, optimizer, device, scaler, num_classes, cfg, ema, is_train, accum_steps):
    model.train(is_train)
    total_loss = total_top1 = total_top5 = 0.0
    total_samples = 0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    total_batches = len(loader)
    progress_every = max(1, int(getattr(cfg, "progress_every", 100)))

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            if is_train:
                images_mixed, soft_targets = mixup_cutmix(
                    images, targets, cfg.mixup_alpha, cfg.cutmix_alpha, cfg.mix_probability, num_classes
                )
                logits = model(images_mixed)
                loss = criterion(logits, soft_targets)
            else:
                logits = model(images)
                loss = criterion(logits, targets)

            loss = loss / accum_steps

        if is_train:
            scaler.scale(loss).backward()
            if step % accum_steps == 0 or step == total_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model)

        with torch.no_grad():
            top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_samples += bs
        total_loss += loss.item() * accum_steps * bs
        total_top1 += float(top1.item()) * bs
        total_top5 += float(top5.item()) * bs

        if step % progress_every == 0 or step == total_batches:
            avg_loss = total_loss / max(1, total_samples)
            avg_top1 = total_top1 / max(1, total_samples)
            phase = "train" if is_train else "val"
            print(f"  [{phase}] batch {step}/{total_batches} | avg_loss={avg_loss:.4f} | avg_top1={avg_top1:.4f}")

    return {
        "loss": total_loss / max(1, total_samples),
        "top1": total_top1 / max(1, total_samples),
        "top5": total_top5 / max(1, total_samples),
    }


# New AMP scaler API (optional immediate override in runtime)
# This line ensures the newer torch.amp API is used when training starts.
scaler = torch.amp.GradScaler(device="cuda", enabled=(device.type == "cuda"))

In [4]:
# AMP compatibility patch: removes deprecation warnings from older torch.cuda.amp calls
# Run this once before the training cell.

if hasattr(torch, "amp") and hasattr(torch, "cuda") and hasattr(torch.cuda, "amp"):
    def _amp_grad_scaler_compat(*args, **kwargs):
        kwargs.setdefault("device", "cuda")
        return torch.amp.GradScaler(*args, **kwargs)

    def _amp_autocast_compat(*args, **kwargs):
        kwargs.setdefault("device_type", "cuda")
        return torch.amp.autocast(*args, **kwargs)

    torch.cuda.amp.GradScaler = _amp_grad_scaler_compat
    torch.cuda.amp.autocast = _amp_autocast_compat
    print("AMP compatibility patch active (torch.cuda.amp -> torch.amp).")
else:
    print("torch.amp not available in this environment; no patch applied.")

AMP compatibility patch active (torch.cuda.amp -> torch.amp).


## 3) Run Training (Best Settings)

Run this cell to start the **real high-accuracy training**.

Tip:
- Keep `head_epochs=2` and `ft_epochs=24` first.
- If you still have GPU time after that, increase `ft_epochs` to 28 or 32 for a final push.

In [5]:
# Permanent fix for end-of-training KeyError('label') in eval split
# Run this once before the training cell.

def _ensure_style_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "relative_path" in out.columns and "label" in out.columns:
        out["label"] = out["label"].astype(int)
        return out[["relative_path", "label"]]

    if out.shape[1] >= 2:
        out = out.iloc[:, :2].copy()
        out.columns = ["relative_path", "label"]
        out["label"] = out["label"].astype(int)
        return out

    raise ValueError("Expected at least 2 columns for style CSV rows.")


def make_eval_split(val_df: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    val_df = _ensure_style_columns(val_df)

    sampled_parts = []
    for _, grp in val_df.groupby("label", sort=False):
        if len(grp) > 1:
            sampled_parts.append(grp.sample(frac=0.5, random_state=seed))
        else:
            sampled_parts.append(grp)

    eval_test = pd.concat(sampled_parts, axis=0)
    eval_val = val_df.drop(index=eval_test.index)

    if len(eval_val) == 0:
        eval_val = val_df.sample(frac=0.5, random_state=seed)
        eval_test = val_df.drop(index=eval_val.index)

    eval_val = _ensure_style_columns(eval_val).reset_index(drop=True)
    eval_test = _ensure_style_columns(eval_test).reset_index(drop=True)
    return eval_val, eval_test

print("Split fix active: make_eval_split now preserves ['relative_path', 'label'] safely.")

Split fix active: make_eval_split now preserves ['relative_path', 'label'] safely.


In [6]:
# Quick sanity check (no training): verifies split keeps required columns
wikiart_dir, train_csv, val_csv = discover_paths(Path.cwd())
val_df_check = filter_existing_rows(load_style_csv(val_csv), wikiart_dir, "val-check")
v1, v2 = make_eval_split(val_df_check, seed=42)
print("val columns:", list(v1.columns), "| test columns:", list(v2.columns))
assert "label" in v1.columns and "label" in v2.columns
print("Sanity check passed: no missing 'label' in eval split.")

val columns: ['relative_path', 'label'] | test columns: ['relative_path', 'label']
Sanity check passed: no missing 'label' in eval split.


In [7]:
project_root = Path.cwd()

cfg = TrainConfig(
    model_name="vit_base_patch16_384.augreg_in21k_ft_in1k",
    image_size=384,
    batch_size=12,
    effective_batch_size=48,
    head_epochs=2,
    ft_epochs=24,
    patience=8,
    warmup_epochs=2,
    head_lr=6e-4,
    ft_backbone_lr=1e-5,
    ft_head_lr=3e-5,
    weight_decay=8e-5,
    mixup_alpha=0.4,
    cutmix_alpha=0.8,
    mix_probability=0.75,
    label_smoothing=0.12,
    ema_decay=0.9997,
    grad_clip_norm=1.0,
    tta=True,
    use_weighted_sampler=True,
    seed=42,
    num_workers=0,  # safer on Windows
)

history_df, final_val, final_test = fit(project_root, cfg)

[train] Filtered out 2 missing files.
Train/Val/Test sizes: 57023/12213/12208
Classes: 27
  [train] batch 100/4751 | avg_loss=4.2821 | avg_top1=0.0358
  [train] batch 200/4751 | avg_loss=4.0934 | avg_top1=0.0521
  [train] batch 300/4751 | avg_loss=3.9735 | avg_top1=0.0558
  [train] batch 400/4751 | avg_loss=3.8721 | avg_top1=0.0644
  [train] batch 500/4751 | avg_loss=3.7884 | avg_top1=0.0690
  [train] batch 600/4751 | avg_loss=3.7138 | avg_top1=0.0729
  [train] batch 700/4751 | avg_loss=3.6453 | avg_top1=0.0794
  [train] batch 800/4751 | avg_loss=3.5972 | avg_top1=0.0832
  [train] batch 900/4751 | avg_loss=3.5416 | avg_top1=0.0905
  [train] batch 1000/4751 | avg_loss=3.4948 | avg_top1=0.0968
  [train] batch 1100/4751 | avg_loss=3.4503 | avg_top1=0.1021
  [train] batch 1200/4751 | avg_loss=3.4140 | avg_top1=0.1060
  [train] batch 1300/4751 | avg_loss=3.3771 | avg_top1=0.1108
  [train] batch 1400/4751 | avg_loss=3.3484 | avg_top1=0.1131
  [train] batch 1500/4751 | avg_loss=3.3163 | avg_t

## 4) Quick Result Check (Simple View)

This gives a clean summary after training.

In [8]:
print("History rows:", len(history_df))
if len(history_df) > 0:
    best_idx = history_df["val_top1"].idxmax()
    print("\nBest validation row:")
    display(history_df.loc[[best_idx]])

    print("\nLast 5 epochs:")
    display(history_df.tail(5))

print("\nFinal Validation:", final_val)
print("Final Test:", final_test)

results_dir = Path.cwd() / "models" / "results"
print("\nSaved files:")
print("-", results_dir / "wikiart_test6_history.csv")
print("-", results_dir / "wikiart_tests_1_to_6_summary.csv")
print("-", Path.cwd() / "models" / "wikiart_test6_best.pt")

History rows: 26

Best validation row:


,epoch,stage,lr,train_loss,train_top1,train_top5,val_loss,val_top1,val_top5
23,24,fine-tune,2.025351e-07,0.529476,0.730127,0.9354,1.888643,0.758536,0.975845



Last 5 epochs:


,epoch,stage,lr,train_loss,train_top1,train_top5,val_loss,val_top1,val_top5
21,22,fine-tune,7.937323e-07,0.549142,0.728355,0.937013,1.882296,0.757635,0.976173
22,23,fine-tune,4.518400e-07,0.525412,0.730864,0.934961,1.884430,0.758208,0.976337
23,24,fine-tune,2.025351e-07,0.529476,0.730127,0.935400,1.888643,0.758536,0.975845
24,25,fine-tune,5.089279e-08,0.521355,0.738844,0.938644,1.889355,0.758372,0.975764
25,26,fine-tune,0.000000e+00,0.510602,0.729899,0.933347,1.889805,0.757799,0.975108



Final Validation: {'loss': 1.8886430350781307, 'top1': 0.7585360057902728, 'top5': 0.9758454159184012}
Final Test: {'loss': 1.9269717161886657, 'top1': 0.7486894035108905, 'top5': 0.9720675026214295}

Saved files:
- c:\Users\Thijs\Desktop\Ai Art Critic\models\results\wikiart_test6_history.csv
- c:\Users\Thijs\Desktop\Ai Art Critic\models\results\wikiart_tests_1_to_6_summary.csv
- c:\Users\Thijs\Desktop\Ai Art Critic\models\wikiart_test6_best.pt
